# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davidsoooooon/ML-Intern---Briones/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My chosen lane is **Lane 2: Refresh / Content Opportunity Scoring**.

I frame this as a **ranking/scoring task** because the main decision is which
content pages should enter the editor's review queue first.

The output will be a ranked list of anonymized pages with a priority score.
A content editor can use that ranking to decide whether to refresh, expand,
protect, consolidate, monitor, or leave each page unchanged.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter proxy target is `is_declining_label`.

A value of `1` means the page's measured trend direction is down, while `0`
means it is not currently classified as down. The model would produce a score
for each page, and the pages with the highest scores would be reviewed first.

This is a **defined same-snapshot proxy**, not a confirmed future outcome.
A stronger final target would measure whether a page declines or improves in
a later time window using only earlier data as features.

Because the proxy comes from `trend_direction` and `trend_pct`, those two
columns must never be used as model features.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is **Precision@50**.

Precision@50 measures how many of the first 50 recommended pages are actual
declining-page proxy cases.

For this project, I will initially define a good result as
`Precision@50 >= 0.70`, meaning at least 35 of the top 50 recommendations are
relevant. It should also perform better than the transparent rule-based
baseline.

This metric matches the real action because an editor has limited time and
will review only the highest-priority pages.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
import pandas as pd

data_url = (
    "https://raw.githubusercontent.com/"
    "Davidsoooooon/ML-Intern---Briones/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

# Starter proxy target.
# Do not use trend_direction or trend_pct as model features later.
df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Lane slice: pages with enough measurable activity to review.
lane_slice = df.loc[
    (df["impressions_90d"] >= 100)
    & (df["sessions_90d"] > 0),
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "ctr",
        "engagement_rate",
        "days_since_last_update",
        "is_declining_label",
    ],
].copy()

print(f"Full dataset: {len(df):,} pages")
print(f"Lane slice: {len(lane_slice):,} pages")
print("Unit of analysis: one row = one anonymized content page")

lane_slice.head(10)

Full dataset: 30,000 pages
Lane slice: 22,006 pages
Unit of analysis: one row = one anonymized content page


,content_id,impressions_90d,clicks_90d,avg_position,ctr,engagement_rate,days_since_last_update,is_declining_label
0,content_304f48230142,3803,29,10.6,0.76,5.88,20,1
1,content_a1fb4e703a9e,15320,7,20.3,0.05,0.00,25,1
2,content_9aa793d4d895,12581,11,36.5,0.09,0.00,20,1
3,content_331d6c4de07b,11751,58,6.2,0.49,1.28,22,0
4,content_d99b7a2d90ca,19140,24,44.0,0.13,0.00,14,1
5,content_d4084a4bc775,3970,1,8.5,0.03,0.00,20,1
7,content_a63219c6e95a,1724,1,21.2,0.06,3.57,22,0
8,content_5e6c160719bc,32574,29,46.0,0.09,5.88,20,1
9,content_c27558df2b0c,1240,2,4.9,0.16,0.00,104,1
10,content_d8ee6cc6d642,20919,324,2.2,1.55,6.75,104,0


The lane slice contains **22,006 anonymized content pages** with at least
100 impressions and more than 0 sessions during the measured period.

Each row represents one content page. The columns describe its visibility,
search performance, engagement, freshness, and starter proxy label.

The `is_declining_label` column is only a same-snapshot proxy. A value of `1`
means the page is currently marked as trending down, while `0` means it is not.
It should be used as the target, not as an input feature.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could recommend every page that is old and receives many
impressions. However, this may send healthy pages for unnecessary edits or
miss newer pages that are already losing useful visibility.

The review priority may depend on several interacting signals, including
impressions, clicks, average position, CTR, engagement, and time since the
last update. These relationships can be too complicated for one universal
if-statement.

ML is useful only if it creates a better ranked review queue than a transparent
rule-based baseline. The output supports the editor's decision; it does not
automatically decide that a page must be changed.

A wrong high-priority recommendation wastes editorial time, while a missed
recommendation may leave an important declining page unnoticed.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.